In [1]:
import string
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

Matplotlib is building the font cache; this may take a moment.


In [2]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB

In [4]:
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

In [11]:
data = {
    'texte': [
        'Félicitations ! Vous avez gagné un prix de 1000 euros.',
        'Bonjour, pourrais-tu m envoyer le compte rendu ?',
        'URGENT : Votre compte bancaire est bloqué.',
        'Salut, on se voit ce soir pour le match de football ?',
        'Offre exceptionnelle ! Achetez une montre gratuite.',
        'N oublie pas d acheter du lait en rentrant.',
        'Gagnez de l argent rapidement sans effort !',
        'Le projet avance bien, nous serons prêts demain.',
        'GAGNANT ! Réclamez votre bon d achat de 500€.',
        'Est-ce que tu as terminé le devoir de ML ?',
        'Promo exclusive : Obtenez 50% de réduction immédiate !',
        'Tu viens à la réunion de 14h dans la grande salle ?',
        'Gagnez un voyage aux Maldives en cliquant ici !',
        'Merci pour ton aide sur ce projet, c était super.',
        'Alerte sécurité : Réinitialisez votre mot de passe.',
    ],
    'label': [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1],
}

In [12]:
df = pd.DataFrame(data)

In [13]:
stop_words = set(stopwords.words('french'))
lemmatizer = WordNetLemmatizer()

In [14]:
def nettoyer_texte(texte):
    # 1. Mettre en minuscules
    texte = texte.lower()
    # 2. Supprimer la ponctuation
    texte = texte.translate(str.maketrans('', '', string.punctuation))
    # 3. Tokenisation simple par espace
    mots = texte.split()
    # 4. Suppression des stop words et Lemmatisation
    mots_nettoyes = [
        lemmatizer.lemmatize(mot) for mot in mots if mot not in stop_words
    ]

    return ' '.join(mots_nettoyes)

In [15]:

df['texte_nettoye'] = df['texte'].apply(nettoyer_texte)

print("--- Aperçu des données après nettoyage ---")
print(df[['texte', 'texte_nettoye', 'label']].head())

vectorizer = TfidfVectorizer()
X_vect = vectorizer.fit_transform(df['texte_nettoye'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X_vect, y, test_size=0.20, random_state=42, stratify=y
)

print(
    f'\nTaille entraînement : {X_train.shape[0]} | Taille test : {X_test.shape[0]}'
)

modeles = {
    'Régression Logistique': LogisticRegression(random_state=42),
    'Naive Bayes (Multinomial)': MultinomialNB(),
    'Random Forest': RandomForestClassifier(random_state=42),
}

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('\n--- Résultats de la Validation Croisée (5-Fold) sur Train ---')
resultats = {}

for nom_modele, modele in modeles.items():
    # Calcul des scores sur les 5 folds
    scores = cross_val_score(
        modele, X_train, y_train, cv=kfold, scoring='accuracy'
    )
    resultats[nom_modele] = scores
    print(
        f'{nom_modele:25s} -> Accuracy Moyenne: {scores.mean():.4f} (+/- std: {scores.std():.4f})'
    )

--- Aperçu des données après nettoyage ---
                                               texte  \
0  Félicitations ! Vous avez gagné un prix de 100...   
1   Bonjour, pourrais-tu m envoyer le compte rendu ?   
2         URGENT : Votre compte bancaire est bloqué.   
3  Salut, on se voit ce soir pour le match de foo...   
4  Offre exceptionnelle ! Achetez une montre grat...   

                                  texte_nettoye  label  
0            félicitations gagné prix 1000 euro      1  
1       bonjour pourraistu envoyer compte rendu      0  
2                 urgent compte bancaire bloqué      1  
3                salut voit soir match football      0  
4  offre exceptionnelle achetez montre gratuite      1  

Taille entraînement : 12 | Taille test : 3

--- Résultats de la Validation Croisée (5-Fold) sur Train ---
Régression Logistique     -> Accuracy Moyenne: 0.4333 (+/- std: 0.0816)
Naive Bayes (Multinomial) -> Accuracy Moyenne: 0.4333 (+/- std: 0.0816)
Random Forest             -